## PILLAR 4: Text Processing & Log Analysis

**Support context:** this is the pillar interviewers actually test. "Find the top N errors," "extract just the PID column," "replace this value across a config" — these are the exact skills that separate someone who can _read_ a log from someone who can _mine_ one fast, under pressure, mid-incident.

---

### `grep` — the flags beyond what you already use

You know `-i`, basic pattern search, and piping into `grep`. Here's the rest:

|Flag|Effect|
|---|---|
|`-v`|**invert** — show lines that _don't_ match|
|`-n`|show line **numbers**|
|`-c`|just the **count** of matching lines, not the lines themselves|
|`-A3`|show 3 lines **after** each match (context)|
|`-B3`|show 3 lines **before** each match|
|`-C3`|3 lines of context on **both** sides|
|`-E`|**extended regex** — lets you use `+`, `?`, `|
|`--color`|highlight the match (often on by default via alias)|

```bash
grep -c "Failed password" /var/log/auth.log
grep -n -A2 -B2 "error" /var/log/syslog
grep -Ev "systemd|CRON" /var/log/syslog | head
```

**TICKET SCENARIO:** _"Something crashed, but the error line alone doesn't explain why."_ `grep -B5 -A5 "Segmentation fault" logfile` — pulls the 5 lines before and after the crash, giving you the lead-up and aftermath in one shot instead of manually scrolling.

---

### `cut` — extract columns from delimited text

**SYNTAX:** `cut -d'delimiter' -f field_number(s)`

In [ ]:
cut -d: -f1 /etc/passwd | head
cut -d: -f1,3 /etc/passwd | head

**READ THE OUTPUT:** `/etc/passwd` is colon-delimited (`root:x:0:0:root:/root:/bin/bash`) — `-f1` pulls just usernames, `-f1,3` pulls username + UID. `cut` is fast and simple but **only works on clean, fixed-delimiter data** — the moment spacing is irregular (like `ps aux` output, which uses variable-width whitespace), `cut` breaks down and you need `awk` instead.

---

### `awk` — column extraction from irregular/whitespace-separated output

**PURPOSE (support):** unlike `cut`, `awk` splits on _any amount_ of whitespace by default, making it the right tool for command output like `ps`, `df`, `ls -l` — anything humans formatted with variable spacing for readability.

In [ ]:
ps aux | awk '{print $2, $11}'

**READ THE OUTPUT:** `$1`, `$2`, ... are positional columns; `$0` is the whole line. This prints just PID (`$2`) and command (`$11`) from `ps aux` — exactly the "get only the PID column" scenario your curriculum calls out by name.

In [ ]:
ps aux | awk '$3 > 0.0 {print $2, $3, $11}'

**READ THE OUTPUT:** `awk` supports a `condition {action}` pattern — this line only prints processes where column 3 (%CPU) is above `0.0`. That's a real, live "find what's actually using CPU right now" one-liner.

**Developer bridge:** think of `awk '{print $2}'` as the shell equivalent of `line.split()[1]` in Python, and `awk '$3 > 0.0 {...}'` as `if float(fields[2]) > 0.0:` — same idea, just column-oriented and streaming line-by-line instead of loading everything into memory.

---

### `sed` — stream editor: find and replace

**SYNTAX:** `sed 's/pattern/replacement/flags'`

In [ ]:
echo "hello world" | sed 's/world/linux/'
sed -n '5,10p' /etc/passwd

**FLAGS:** `s///` = substitute (first match per line by default, add `g` for **all** matches per line). `-n '5,10p'` = print only lines 5-10 (suppresses default auto-print, then explicitly prints that range) — a quick way to view a specific slice of a huge file without opening it.

In [ ]:
sed 's/dennie/REDACTED/g' /etc/passwd

**GOTCHA:** that command prints the _modified_ result to your terminal — it does **not** touch the actual file unless you add `-i` (in-place edit). `sed -i` is powerful and has zero undo — always test without `-i` first, and consider `sed -i.bak` (creates a backup automatically) when editing anything that matters.

**TICKET SCENARIO:** _"This config has the wrong IP address hardcoded in 40 places."_ `sed -i 's/10.0.0.5/10.0.0.12/g' /etc/someconfig.conf` — instant, precise, scriptable — versus manually opening it in an editor and hunting.

---

### `tr` — translate or delete characters

In [ ]:
echo "Hello World" | tr 'a-z' 'A-Z'
cat /etc/passwd | tr ':' '\t' | head

**PURPOSE (support):** character-level, not line/word-level — good for quick reformatting, like turning a delimiter into something more readable, or normalizing case before a comparison.

---

### `xargs` — pass piped output as arguments to another command

**PURPOSE (support):** many commands (`find`, `grep -l`) print a _list of things_, but don't act on them — `xargs` takes that list and feeds it as arguments into a second command.

In [ ]:
find /tmp -name "*.txt" | xargs rm -v

**GOTCHA — the classic footgun:** filenames with spaces break naive `xargs` usage (each space looks like a new argument). Safer pattern:

In [ ]:
find /tmp -name "*.txt" -print0 | xargs -0 rm -v

`-print0`/`-0` use a null byte instead of whitespace/newline as the separator — null bytes can't appear in filenames, so this is bulletproof.

> **Support rule: when scripting deletions or bulk operations across unknown filenames, always use the `-print0`/`xargs -0` pair.**

---

### `tee` — write to terminal and a file, simultaneously

In [ ]:
echo "test line" | tee /tmp/output.txt
cat /tmp/output.txt

**TICKET SCENARIO:** you're running a long diagnostic command during a live incident and need to both _watch it happen_ and _keep a record_ for the postmortem: `some_diagnostic_command | tee /tmp/incident_notes.log`. Without `tee`, you'd have to choose between watching it live or capturing it — not both.

---

### The Real Scenario: Top 10 Most Common Errors in a Log

This is the pipeline everything above was building toward:

In [ ]:
sudo grep -i "error\|fail" /var/log/syslog | awk '{$1=$2=$3=$4=""; print $0}' | sort | uniq -c | sort -rn | head -10

**READ THE PIPELINE, stage by stage:**

1. `grep -i "error\|fail"` — pull only lines mentioning errors/failures (case-insensitive)
2. `awk '{$1=$2=$3=$4="";print $0}'` — blank out the first 4 fields (timestamp/hostname/process info, which vary per line and would prevent grouping) so only the _message_ remains
3. `sort` — group identical messages together (required before `uniq`)
4. `uniq -c` — collapse duplicates, prefixed with a count
5. `sort -rn` — biggest offenders first
6. `head -10` — top 10 only

**TICKET SCENARIO:** _"The app has been unstable all night, where do I even start?"_ This one line turns thousands of scattered log lines into a ranked top-10 list of what's actually recurring — the difference between "I have no idea" and "here are the 3 things happening over and over, let's start there."

---

### TRY NOW

In [ ]:
dennie@HP:~/linux-tut$ grep -c "Failed password" /var/log/auth.log
0

In [ ]:
dennie@HP:~/linux-tut$ ps aux | awk '$3 > 0.0 {print $2, $3, $11}'
411 0.7 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
460 5.7 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
471 0.2 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
482 0.5 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
526 0.4 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
724 2.9 /home/dennie/.vscode-server/extensions/anthropic.claude-code-2.1.246-linux-x64/resources/native-binary/claude
3771 0.1 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
4576 0.1 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
4586 0.1 /home/dennie/.vscode-server/bin/110a328ea54b42367b803ec53ee0bf52ef26b419/node
5757 0.1 /usr/libexec/wsl-pro-service

In [ ]:
dennie@HP:~/linux-tut$ sudo grep -i "error\|fail" /var/log/syslog | awk '{$1=$2=$3=$4="";print $0}' | sort | uniq -c | sort -rn | head -10
[sudo] password for dennie: 
      8     to connect to bus: No such file or directory
      8     - Process error reports when automatic reporting is enabled (timer based) was skipped because of an unmet condition check (ConditionPathExists=/var/lib/apport/autoreport).
      8     - Process error reports when automatic reporting is enabled (file watch) was skipped because of an unmet condition check (ConditionPathExists=/var/lib/apport/autoreport).
      7     to register legacy timer interrupt
      7     Underlying Hyper-V does not support order less than 9. Hypercall failed
      7     Correctable Errors collector initialized.
      5     (107) ERROR: CheckConnection: getaddrinfo() failed: -5
      3     Cold memory discard hypercall failed with status e00000005
      3     Cold memory discard hypercall failed with status 1800000005
      3     (109) ERROR: CheckConnection: getaddrinfo() failed: -5